# Version control: branching, commits, and change requests

Istari **systems** group resources and files together in a way that is version controlled
(generally available in the 07.2026 release). This recipe drives the full workflow from the
official Python client (**`istari-digital-client` ≥ 10.14.0**), the same flow you'd click
through in the UI:

1. get a system's **branches** (every system starts with a `baseline` branch),
2. **branch** to isolate in-progress work,
3. **commit** files to the branch — each commit points at a **snapshot**, a complete picture
   of every tracked file and subsystem at that moment,
4. open a **change request** from the working branch into the branch you want to update,
5. review the delta (**added / removed / changed**), then **merge** — or close without merging.

The notebook is self-contained: it creates a scratch system and sample files, and archives
the system again at the end.

### Prerequisites

- **`istari-digital-client`** (10.14.0 or later) and **`python-dotenv`**
- `ISTARI_REGISTRY_URL` and `ISTARI_PERSONAL_ACCESS_TOKEN` set in the environment or in
  `samples/.env` — create a personal access token under **Settings → Developer settings**
  in the Istari web app.

> **Docs:** [Key Concepts](https://docs.istaridigital.com/intro/key-concepts) ·
> [Terminology](https://docs.istaridigital.com/intro/terminology) ·
> [Python Client — Quick Start](https://docs.istaridigital.com/developers/SDK/setup)

## 1 · Connect and verify

Build `Configuration` from the standard env vars and construct `Client`. Any lightweight
call (here `list_systems`) proves the token works.

In [1]:
import json
import os
from pathlib import Path

import dotenv
from istari_digital_client import Client, Configuration

dotenv.load_dotenv()
_registry_url = os.environ.get("ISTARI_REGISTRY_URL")
_token = os.environ.get("ISTARI_PERSONAL_ACCESS_TOKEN")
if not _registry_url or not _token:
    raise RuntimeError("Set ISTARI_REGISTRY_URL and ISTARI_PERSONAL_ACCESS_TOKEN (e.g. in samples/.env)")

client = Client(Configuration(registry_url=_registry_url, registry_auth_token=_token))

print("Systems visible:", len(client.list_systems().items))

2026-07-23 13:13:55 -  istari_digital_client.compatibility:process_response_headers:69 - INFO - Connected to Istari Registry v10.21.2 — SDK is up to date.


Systems visible: 10


## 2 · Create a scratch system

So the recipe never touches real work, we make a throwaway system to branch and merge in.
(Adapting this to your own system is a one-line change: `system = client.get_system("<system-id>")`.)

In [2]:
from istari_digital_client import NewSystem

system = client.create_system(NewSystem(
    name="Cookbook: version-control recipe",
    description="Scratch system created by branching-and-change-requests.ipynb — safe to archive.",
))
system = client.get_system(system.id)
print("System:", system.id)

System: fc623d48-e551-4e1f-bf69-c9d07ba766d5


## 3 · Get the system's branches

Branch operations live on the `System` object. Every system starts with a **`baseline`**
branch, which you can look up by name. `list_branches()` returns all *other* branches —
freshly created systems have none yet.

In [3]:
baseline = system.get_branch("baseline")
branches = system.list_branches()   # all branches except baseline

print("baseline branch id:", baseline.id)
print("other branches:", [b.tag for b in branches])

baseline branch id: 759d86ed-5c5f-4c22-af56-1b73e5e60f5f
other branches: []


## 4 · Create a branch

Create a new branch off of any existing branch to isolate the changes you're making —
engineers working on separate branches don't impact each other's work.

> **Note:** the `add_to_branch` helper commits to *working* branches; to update `baseline`,
> merge a change request into it (sections 6–8).

In [4]:
v2 = system.create_branch("v2", from_branch=baseline)
print("Created branch:", v2.tag, v2.id)

Created branch: v2 afd76bce-093e-4a4f-be54-56f94ea501f6


## 5 · Commit files to the branch

Upload a file with `add_file`, then commit its **revision** to the branch. You can commit
several revisions (and subsystems) in a single commit. Each call to `add_to_branch` /
`remove_from_branch` creates **one commit**; each commit points at a **snapshot** — the
complete contents of the system at that moment — so `list_branch_history` reads like `git log`.

First, generate two small sample files so the notebook is self-contained:

In [5]:
import struct
import tempfile
import zlib

workdir = Path(tempfile.mkdtemp(prefix="istari-vc-recipe-"))

bom_path = workdir / "bill_of_materials.json"
bom_path.write_text(json.dumps(
    {"assembly": "Group 3 UAS Drone",
     "parts": [{"name": "fuselage", "qty": 1}, {"name": "wing", "qty": 2}]},
    indent=2,
))

def _png_chunk(tag: bytes, data: bytes) -> bytes:
    return struct.pack(">I", len(data)) + tag + data + struct.pack(">I", zlib.crc32(tag + data))

render_path = workdir / "back.png"   # a minimal valid 1x1 PNG stand-in for a CAD render
render_path.write_bytes(
    b"\x89PNG\r\n\x1a\n"
    + _png_chunk(b"IHDR", struct.pack(">IIBBBBB", 1, 1, 8, 2, 0, 0, 0))
    + _png_chunk(b"IDAT", zlib.compress(b"\x00\x24\x36\x7f"))
    + _png_chunk(b"IEND", b"")
)
print("Sample files in", workdir)

Sample files in /var/folders/tx/gcl9q_cj1xj8y10gql78wjwr0000gn/T/istari-vc-recipe-ef1q55ph


In [6]:
render = client.add_file(str(render_path))
bom = client.add_file(str(bom_path))

v2 = system.add_to_branch(v2, revisions=[render.revision, bom.revision])
print("Committed — branch now at snapshot:", v2.snapshot_id)

Committed — branch now at snapshot: ac4877d3-22c2-49ee-b7a0-e82059e48c64


To remove tracked files from a branch, use `system.remove_from_branch()` with the same
arguments. Both operations show up in the branch's commit history (newest first):

In [7]:
v2 = system.remove_from_branch(v2, revisions=[render.revision])
v2 = system.add_to_branch(v2, revisions=[render.revision])   # put it back for the merge

for commit in system.list_branch_history(v2):   # newest first, like `git log`
    print(commit.snapshot_id, commit.created)

a335464d-0cc6-4ff7-a144-faf5ef886512 2026-07-23 17:13:59.252164+00:00
3c97be76-a3b1-432c-8b82-1084800cd69f 2026-07-23 17:13:58.596002+00:00
ac4877d3-22c2-49ee-b7a0-e82059e48c64 2026-07-23 17:13:57.919701+00:00
6ca8a333-1ffa-4079-8df4-f56a7336e9b3 2026-07-23 17:13:56.190315+00:00


## 6 · Create a change request

When you want to merge those changes, create a **change request** from your working branch
(the **source**) into the branch you want to update (the **target**).

In [8]:
from istari_digital_client import ChangeRequestCreateRequest

response = client.create_change_request(
    system_id=system.id,
    change_request_create_request=ChangeRequestCreateRequest(
        source_tag_id=v2.id,
        target_tag_id=baseline.id,
        title="Move v2 to baseline",
        description="Added the renders and bill of materials",
    ),
)

change_request = response.actual_instance   # an OpenChangeRequestResponse
print(change_request.change_request_id, change_request.status)

eaf52794-0376-4620-8851-a9352119d566 OPEN


## 7 · Review the changes

Just like in the UI, you can see which resources and systems were **added**, **removed**,
or **changed** before merging. Get the high-level counts first, then page through the full
delta. You can filter with `component_type=` (`RESOURCE` or `SUBSYSTEM`) and `change_types=`
(a list of `ADDED`, `REMOVED`, `CHANGED`); if `has_more` is true on the returned page, pass
its `next_cursor` back as `cursor=` to fetch the next page.

In [9]:
summary = client.change_request_change_summary(
    system_id=system.id,
    change_request_id=change_request.change_request_id,
)
print(f"{summary.added_count} added, {summary.removed_count} removed, "
      f"{summary.changed_count} changed")

2 added, 0 removed, 0 changed


In [10]:
from istari_digital_client import ChangeRequestComponentType, ChangeRequestDiffType

diff_page = client.change_request_changes(
    system_id=system.id,
    change_request_id=change_request.change_request_id,
    component_type=ChangeRequestComponentType.RESOURCE,          # optional filters
    change_types=[ChangeRequestDiffType.ADDED],
)

for item in diff_page.items:
    diff = item.actual_instance   # a resource diff or a subsystem diff
    print(diff.diff_type, diff.name)

ChangeRequestDiffType.ADDED back.png
ChangeRequestDiffType.ADDED bill_of_materials.json


## 8 · Merge the change request

When you're satisfied with the changes, merge. The target branch now points at the same
contents (the same **snapshot**) as your working branch.

In [11]:
from istari_digital_client import ChangeRequestMergeRequest

merged = client.merge_change_request(
    system_id=system.id,
    change_request_id=change_request.change_request_id,
    change_request_merge_request=ChangeRequestMergeRequest(
        comment="Approved and merged",
    ),
).actual_instance

print(merged.status)

baseline = system.get_branch("baseline")
assert baseline.snapshot_id == v2.snapshot_id
print("baseline now at snapshot:", baseline.snapshot_id)

MERGED


baseline now at snapshot: a335464d-0cc6-4ff7-a144-faf5ef886512


## 9 · Or close without merging

To reject a change request instead of merging it, set its status to `CLOSED` with
`update_change_request`. (There is no separate `close_change_request` method on the client.)

In [12]:
from istari_digital_client import ChangeRequestStatus, ChangeRequestUpdateRequest

# open a second, throwaway change request just to demonstrate closing
scratch = system.create_branch("close-demo", from_branch=v2)
scratch = system.add_to_branch(scratch, revisions=[client.add_file(str(bom_path)).revision])

cr2 = client.create_change_request(
    system_id=system.id,
    change_request_create_request=ChangeRequestCreateRequest(
        source_tag_id=scratch.id,
        target_tag_id=v2.id,
        title="Demonstrate closing",
    ),
).actual_instance

closed = client.update_change_request(
    system_id=system.id,
    change_request_id=cr2.change_request_id,
    change_request_update_request=ChangeRequestUpdateRequest(
        status=ChangeRequestStatus.CLOSED,
        comment="Not needed",
    ),
).actual_instance

print(closed.status)

CLOSED


## 10 · Clean up

Archive the scratch system (reversible with `client.restore_system`). Skip this cell if you
want to poke around the system in the UI first — the commit history and both change requests
are all visible there.

In [13]:
client.archive_system(system_id=system.id)
print("Archived scratch system", system.id)

Archived scratch system fc623d48-e551-4e1f-bf69-c9d07ba766d5
